# Running Neurodesk on HPC with Singularity/Apptainer

**Author**: Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

Neurodesk tools are distributed as **Singularity/Apptainer containers**, making them easy to deploy on high-performance computing (HPC) clusters. This tutorial teaches you how to use Neurodesk tools on HPC systems via CVMFS (a global filesystem), submit large-scale batch jobs to SLURM, and parallelize analyses across hundreds of subjects. By the end, you will be able to run production-grade neuroimaging pipelines on your institution's compute resources.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Access Neurodesk containers on HPC via CVMFS or by pulling container images
- Load Neurodesk tools using the Lmod module system (`ml` command)
- Start interactive sessions on compute nodes with `srun`
- Write and submit SLURM batch scripts to process large datasets
- Create array jobs to parallelize processing across subjects
- Monitor and troubleshoot HPC jobs
:::

## Citation and Resources

### Tools and platforms used in this workflow

__Singularity/Apptainer__
: Singularity is a container platform designed for HPC and scientific computing. Apptainer is the modern successor to Singularity. [https://apptainer.org/](https://apptainer.org/)

__CVMFS__
: CernVM FileSystem enables transparent, on-demand access to container images across HPC clusters worldwide. [https://cernvm.cern.ch/fs/](https://cernvm.cern.ch/fs/)

__Neurodesk__
: A containerized neuroimaging toolkit with pre-installed analysis software. [https://neurodesk.org](https://neurodesk.org)

### Educational resources

- [Neurodesk documentation](https://neurodesk.org)
- [SLURM job submission guide](https://slurm.schedmd.com/sbatch.html)
- [Apptainer User Guide](https://apptainer.org/docs/)
- [HPC best practices for neuroimaging](https://neurodesk.org/tutorials/)

## Prerequisites

:::{admonition} Before you begin
:class: warning
You will need:
- A user account on an HPC cluster with SLURM job scheduler
- SSH access to the login node
- At least basic familiarity with the Linux command line
- Your HPC cluster must have either CVMFS mounted or permission to pull Singularity images

Check with your system administrator if you are unsure whether your HPC has CVMFS or Singularity/Apptainer installed.
:::

- [x] HPC cluster access with SLURM
- [ ] Terminal or SSH client to access the HPC
- [ ] Familiarity with SLURM commands (`sbatch`, `squeue`, `srun`)
- [ ] (Optional) A BIDS dataset or neuroimaging data ready for batch processing
- [ ] (Optional) Access to a shared storage location (e.g., `/projects/`, `/scratch/`)

## Overview: Neurodesk on HPC

Neurodesk tools are distributed as **Singularity/Apptainer containers**. On HPC clusters with **CVMFS** (CernVM FileSystem) enabled, you can access these containers globally without downloading them individually. This approach enables:

- **Portability**: The same analysis code runs on your laptop and your institution's supercomputer
- **Reproducibility**: Containers freeze software versions, preventing "works on my machine" problems
- **Scalability**: Launch jobs across hundreds of compute nodes simultaneously
- **Ease**: Use the familiar `ml` (module load) command to access tools, just like on local Neurodesk

### The Architecture

![HPC architecture with Neurodesk containers](/static/tutorials/about_neurodesk/hpc/hpc_architecture.png)
*Neurodesk tools distributed via CVMFS on HPC: login nodes schedule jobs, compute nodes pull containers from the global CVMFS cache.*

## Checking Availability on Your HPC

Before writing batch scripts, verify that your HPC cluster has the tools needed to run Neurodesk.

### Check 1: Is CVMFS Available?

SSH into your HPC cluster and list the CVMFS mount:

```bash
ls /cvmfs/neurodesk.ardc.edu.au/
```

If this command succeeds and shows a list of containers, CVMFS is working. Example output:

```
containers/
neurodesk_setup.sh
etc/
```

If you see "No such file or directory", CVMFS is not mounted. Contact your system administrator or see the "Fallback: Pulling Container Images" section below.

### Check 2: Is Lmod (Module System) Available?

```bash
ml avail
```

If this shows a list of available modules (like `fsl`, `ants`, `dcm2niix`), your cluster has Lmod configured. If you see "command not found" or "ml: command not found", the module system is not set up.

**Fallback:** You can still access Neurodesk without Lmod by directly sourcing the container:

```bash
source /cvmfs/neurodesk.ardc.edu.au/neurodesk_setup.sh
ml fsl/6.0.7.8
```

### Check 3: Is Singularity/Apptainer Installed?

```bash
which apptainer
# or
which singularity
```

CVMFS works automatically with Singularity/Apptainer, so if CVMFS is mounted, the container runtime is almost certainly available.

### Fallback: Manual Neurodesk Setup

If CVMFS is not available, you can set up Neurodesk manually:

```bash
# Download and extract Neurodesk setup
curl -fsSL https://raw.githubusercontent.com/NeuroDesk/neurodesk/main/setup_neurodesk.sh | bash
source ~/.bashrc
```

This installs Neurodesk in your home directory (~/.neurodesk). Consult the [Neurodesk GitHub](https://github.com/NeuroDesk/neurodesk) for details.

## Interactive Job with `srun`

For quick testing or development, start an **interactive session** on a compute node using `srun`. This allocates resources and drops you into a shell on a compute node where you can run commands interactively.

### Starting an Interactive Session

```bash
srun --ntasks=1 --cpus-per-task=4 --mem=16G --time=2:00:00 --pty bash
```

This command:
- `--ntasks=1`: Request 1 task (don't parallelize)
- `--cpus-per-task=4`: Allocate 4 CPU cores
- `--mem=16G`: Allocate 16 GB RAM
- `--time=2:00:00`: Reserve for 2 hours (HH:MM:SS format)
- `--pty bash`: Open an interactive bash shell

After a few seconds, SLURM reserves the resources and you are on a compute node. Load Neurodesk tools:

```bash
ml fsl/6.0.7.8
```

Run your analysis interactively:

```bash
bet input.nii.gz output_brain.nii.gz -R -f 0.5
```

When done, exit the session:

```bash
exit
```

### Common `srun` Options

| Flag | Purpose | Example |
|------|---------|----------|
| `--ntasks` | Number of parallel tasks | `--ntasks=4` for 4 parallel processes |
| `--cpus-per-task` | CPU cores per task | `--cpus-per-task=8` for 8-core parallelization |
| `--mem` | Total RAM to allocate | `--mem=64G` for 64 GB |
| `--time` | Wall-clock time limit | `--time=1:30:00` for 1.5 hours |
| `--partition` | Compute partition (queue) | `--partition=gpu` for GPU partition |
| `--gres` | Generic resources (GPUs) | `--gres=gpu:1` for 1 GPU |
| `--job-name` | Job label | `--job-name=my_analysis` |
| `--pty` | Pseudo-terminal (for interactive) | Always include for interactive sessions |

## Batch Script for a Single Subject

For production analyses, submit a **batch script** to SLURM rather than using interactive sessions. Create a file called `process_subject.sh`:

```bash
#!/bin/bash
#SBATCH --job-name=neurodesk_fsl
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=1:00:00
#SBATCH --output=logs/%x_%j.out
#SBATCH --error=logs/%x_%j.err

# ============================================
# Neurodesk FSL Brain Extraction Example
# Process a single subject's T1w image
# ============================================

# Load Neurodesk module system
source /cvmfs/neurodesk.ardc.edu.au/neurodesk_setup.sh

# Load FSL
ml fsl/6.0.7.8

# Define directories
DATA_DIR=/projects/my-study/data
OUTPUT_DIR=/scratch/my-study/derivatives
SUBJECT=${1:-sub-001}  # Default to sub-001 if not provided

# Create output directory
mkdir -p ${OUTPUT_DIR}/${SUBJECT}/anat

# Run brain extraction with BET
echo "Processing ${SUBJECT}..."
bet ${DATA_DIR}/${SUBJECT}/anat/${SUBJECT}_T1w.nii.gz \
    ${OUTPUT_DIR}/${SUBJECT}/anat/${SUBJECT}_T1w_brain.nii.gz \
    -R -f 0.5

echo "Completed ${SUBJECT}."
```

### Key components:

1. **Shebang** (`#!/bin/bash`): Specifies this is a bash script
2. **SBATCH directives**: All lines starting with `#SBATCH` are job configuration
3. **`source /cvmfs/...`**: Activate Neurodesk module system
4. **`ml fsl/6.0.7.8`**: Load the specific FSL version
5. **Job logic**: Your actual analysis (brain extraction in this example)

### Submitting the script:

```bash
sbatch process_subject.sh sub-001
```

SLURM immediately returns a job ID (e.g., "Submitted batch job 12345") and schedules your job in the queue. Check its status:

```bash
squeue -j 12345
```

Output logs are written to `logs/neurodesk_fsl_12345.out` and `logs/neurodesk_fsl_12345.err`.

## Array Jobs for Multiple Subjects

To process all subjects in parallel, use **SLURM array jobs**. Modify the batch script to add an array directive:

```bash
#!/bin/bash
#SBATCH --job-name=neurodesk_fsl_array
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=1:00:00
#SBATCH --array=1-20              # <-- Run 20 parallel jobs (one per subject)
#SBATCH --output=logs/%x_%a.out   # <-- Use %a for array job index
#SBATCH --error=logs/%x_%a.err

# ============================================
# Neurodesk FSL Array Job for Multiple Subjects
# ============================================

# Load Neurodesk module system
source /cvmfs/neurodesk.ardc.edu.au/neurodesk_setup.sh

# Load FSL
ml fsl/6.0.7.8

# Define directories
DATA_DIR=/projects/my-study/data
OUTPUT_DIR=/scratch/my-study/derivatives

# Create a list of all subjects
SUBJECTS=(sub-001 sub-002 sub-003 sub-004 sub-005 sub-006 sub-007 sub-008 \
          sub-009 sub-010 sub-011 sub-012 sub-013 sub-014 sub-015 sub-016 \
          sub-017 sub-018 sub-019 sub-020)

# Get the subject for this array job index
SUBJECT=${SUBJECTS[$((SLURM_ARRAY_TASK_ID - 1))]}

# Create output directory
mkdir -p ${OUTPUT_DIR}/${SUBJECT}/anat

# Run brain extraction
echo "Processing ${SUBJECT}..."
bet ${DATA_DIR}/${SUBJECT}/anat/${SUBJECT}_T1w.nii.gz \
    ${OUTPUT_DIR}/${SUBJECT}/anat/${SUBJECT}_T1w_brain.nii.gz \
    -R -f 0.5

echo "Completed ${SUBJECT}."
```

### Key differences:

1. **`#SBATCH --array=1-20`**: SLURM creates 20 parallel jobs (indices 1–20)
2. **`SLURM_ARRAY_TASK_ID`**: Each job receives a unique index from 1 to 20
3. **`%a` in output path**: Each job's logs go to separate files (e.g., `neurodesk_fsl_array_1.out`, `neurodesk_fsl_array_2.out`)

### Submitting the array job:

```bash
sbatch process_subjects_array.sh
```

SLURM submits all 20 jobs simultaneously. Check their status:

```bash
squeue -u $USER
```

You will see something like:

```
JOBID PARTITION     NAME     USER ST       TIME  NODES CPUS
12345_1   general     neurodesk... user    R       0:30      1    4
12345_2   general     neurodesk... user    R       0:25      1    4
12345_3   general     neurodesk... user    PD       0:00      1    4
...
```

(ST = status: R = running, PD = pending)

## Monitoring and Managing Jobs

Learn to track and control your SLURM jobs.

### View your jobs

```bash
# List all your jobs
squeue -u $USER

# List a specific job
squeue -j 12345

# List jobs for a specific partition
squeue -p gpu
```

### Detailed job information

```bash
# Get detailed info about a job
scontrol show job 12345

# Example output:
# JobId=12345 JobName=neurodesk_fsl
# UserId=username(1001) GroupId=group(1000)
# Priority=4294901759 Nice=0 Account=default QOS=normal
# JobState=RUNNING Reason=None Dependency=(null)
# ...
```

### Real-time job monitoring

```bash
# Watch squeue output every 2 seconds
watch -n 2 'squeue -u $USER'

# Exit with Ctrl+C
```

### Cancel a job

```bash
# Cancel a single job
scancel 12345

# Cancel all your jobs
scancel -u $USER

# Cancel a specific array job index
scancel 12345_5  # Cancel job 5 from array 12345
```

### Inspect output logs

```bash
# View the last 50 lines of output
tail -50 logs/neurodesk_fsl_12345.out

# Search for errors in output
grep -i error logs/neurodesk_fsl_12345.err

# Stream output in real-time
tail -f logs/neurodesk_fsl_12345.out
```

## Tips and Troubleshooting

### Tip 1: CVMFS Caching — First Run is Slower

The first time you use a Neurodesk container on your HPC, CVMFS downloads it from the global cache. This may take a few minutes. Subsequent runs use the cached copy and are much faster.

**Solution:** Plan for extra time on first runs. For array jobs, the first few tasks may be slower as they download the container.

```bash
# Check CVMFS cache status
du -sh /var/lib/cvmfs/neurodesk.ardc.edu.au/
```

### Tip 2: Always `ml purge` Between Jobs

If you load multiple Neurodesk tools in the same script, **always purge modules first** to avoid conflicts:

```bash
ml purge                # Clear all loaded modules
ml fsl/6.0.7.8          # Load FSL
# Use FSL...

ml purge                # Clear again
ml ants/2.4.3           # Load ANTs
# Use ANTs...
```

Without `ml purge`, loading a second module may inherit paths and variables from the first, causing conflicts.

### Tip 3: Set Memory and Time Appropriately

Underestimating memory or time causes jobs to fail silently:

```bash
# Too little memory — job killed with "Out of memory"
#SBATCH --mem=4G  # Probably too small for neuroimaging

# Better estimate
#SBATCH --mem=16G  # Typical for single-subject processing
```

Check completed job statistics:

```bash
seff 12345  # Shows CPU and memory utilization
```

### Tip 4: Use `$TMPDIR` for Temporary Files on HPC

HPC clusters allocate fast local storage on compute nodes. Use `$TMPDIR` instead of `/tmp`:

```bash
# Create working directory
mkdir -p $TMPDIR/working

# Process data in temporary directory (faster)
fslmaths input.nii.gz -mul 2 $TMPDIR/working/output.nii.gz

# Move final result to shared storage
mv $TMPDIR/working/output.nii.gz /projects/my-study/results/
```

`$TMPDIR` is automatically created on the compute node and cleaned up after your job finishes.

### Tip 5: Dependency Chains for Multi-Stage Pipelines

Process data in stages with job dependencies:

```bash
# Submit preprocessing job
JOB1=$(sbatch preprocess.sh | awk '{print $NF}')

# Submit statistics job only after preprocessing completes
JOB2=$(sbatch --dependency=afterok:$JOB1 stats.sh | awk '{print $NF}')

# Submit visualization job after stats completes
sbatch --dependency=afterok:$JOB2 visualize.sh
```

This ensures jobs run in the correct order without manually resubmitting.

### Troubleshooting: Common Errors

| Error | Cause | Solution |
|-------|-------|----------|
| `command not found` after `ml fsl/...` | CVMFS not mounted or module not found | Check `ls /cvmfs/neurodesk.ardc.edu.au/` and ask admin |
| `Out of memory` | Insufficient RAM allocated | Increase `--mem=` flag |
| `Time limit exceeded` | Job took too long | Increase `--time=` flag |
| Modules conflict (`ml ants` overwrites FSL) | Not purging between loads | Add `ml purge` before each `ml` command |
| Job stuck in queue (PD status) | Insufficient cluster resources | Reduce `--cpus-per-task` or `--mem=` |
| Cannot write to `/projects/` | Permission denied | Check directory ownership: `ls -ld /projects/my-study/` |

## Summary

In this tutorial you learned:

1. **How Neurodesk works on HPC**: Containers distributed via CVMFS, accessed with `ml` commands
2. **Interactive sessions with `srun`**: For testing and development on compute nodes
3. **Batch scripts with `sbatch`**: For production job submission
4. **Array jobs**: For parallel processing of large datasets across subjects
5. **Job monitoring with `squeue` and `scontrol`**: Track and debug your jobs
6. **Best practices**: Use `$TMPDIR`, purge modules, set realistic time/memory limits
7. **Troubleshooting**: Common errors and how to fix them

:::{seealso}
- [Accessing Neurodesk Tools](accessing_neurodesk_tools.ipynb) — how to find available Neurodesk modules
- [Managing Files and Storage](neurodesktop_storage.ipynb) — best practices for file organization on HPC
- [SLURM Documentation](https://slurm.schedmd.com/) — comprehensive SLURM reference
- [Neurodesk Tools](https://neurodesk.org/overview/) — list of pre-installed tools and versions
:::